# **1. Dataset Introduction**


The dataset used is the **Heart Disease Dataset** from the UCI Machine Learning Repository.

- **Source**: [UCI ML Repository - Heart Disease](https://archive.ics.uci.edu/dataset/45/heart+disease)
- **Data Size**: 303 rows, 13 features + 1 target
- **Task**: Binary Classification — predict whether a patient has heart disease (1) or not (0)
- **Features**: Mix of numerical (age, trestbps, chol, thalach, oldpeak) and categorical (sex, cp, fbs, restecg, exang, slope, ca, thal)
- **Missing Values**: Present in `ca` and `thal` columns

# **2. Import Libraries**

In this stage, you need to import several Python libraries required for data analysis and building machine learning or deep learning models.

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

!pip install ucimlrepo -q
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

print("Libraries successfully imported!")

Libraries successfully imported!


# **3. Load Dataset**

In this stage, you need to load the dataset into the notebook. If the dataset is in CSV format, you can use the pandas library to read it. Make sure to check the first few rows of the dataset to understand its structure and ensure the data has been loaded correctly.

If the dataset is on Google Drive, make sure you connect Google Drive to Colab first. Once the dataset is successfully loaded, the next step is to check the data consistency and prepare it for further analysis.

If the dataset consists of unstructured data, please adjust it according to the format of the Machine Learning Development or Applied Machine Learning classes.

In [19]:
# Install package if not already present
!pip install ucimlrepo -q

# Fetch dataset
heart_disease = fetch_ucirepo(id=45)

# Combine features and target
X = heart_disease.data.features
y = heart_disease.data.targets

df = pd.concat([X, y], axis=1)

# Rename target column for clarity
df.rename(columns={'num': 'target'}, inplace=True)

# Convert target to binary (0 = no disease, 1 = disease)
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

print(f"Shape dataset: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

Shape dataset: (303, 14)

First 5 rows:


   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   1       145   233    1        2      150      0      2.3      3   
1   67    1   4       160   286    0        2      108      1      1.5      2   
2   67    1   4       120   229    0        2      129      1      2.6      2   
3   37    1   3       130   250    0        0      187      0      3.5      3   
4   41    0   2       130   204    0        2      172      0      1.4      1   

    ca  thal  target  
0  0.0   6.0       0  
1  3.0   3.0       1  
2  2.0   7.0       1  
3  0.0   3.0       0  
4  0.0   3.0       0  

In [20]:
# Save raw dataset before processing
df_raw = pd.concat([heart_disease.data.features, heart_disease.data.targets], axis=1)
df_raw.to_csv('heart_disease_raw.csv', index=False)
print("Raw dataset saved successfully!")

Raw dataset saved successfully!


In [21]:
print("=== INFO DATASET ===")
print(df.info())

print("\n=== DESCRIPTIVE STATISTICS ===")
df.describe()

=== INFO DATASET ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
 13  target    303 non-null    int64  
dtypes: float64(3), int64(11)
memory usage: 33.3 KB
None

=== DESCRIPTIVE STATISTICS ===


# **4. Exploratory Data Analysis (EDA)**

In this stage, you will perform **Exploratory Data Analysis (EDA)** to understand the characteristics of the dataset.

The goal of EDA is to obtain deep initial insights into the data and determine the next steps in analysis or modeling.

In [22]:
print("=== MISSING VALUES ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage (%)': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

=== MISSING VALUES ===
      Missing Count  Percentage (%)
ca                4        1.320132
thal              2        0.660066


In [23]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='target', data=df, palette='Set2')
plt.title('Target Distribution (0 = No Disease, 1 = Disease)')
plt.xlabel('Target')
plt.ylabel('Count')

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom')
plt.tight_layout()
plt.show()

print(df['target'].value_counts())

target
0    164
1    139
Name: count, dtype: int64


In [24]:
numerical_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

df[numerical_cols].hist(figsize=(12, 8), bins=20, color='steelblue', edgecolor='black')
plt.suptitle('Numerical Feature Distribution', fontsize=14)
plt.tight_layout()
plt.show()

In [25]:
plt.figure(figsize=(12, 8))
correlation = df.corr()
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [26]:
fig, axes = plt.subplots(1, 5, figsize=(18, 5))
for i, col in enumerate(numerical_cols):
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(col)
plt.suptitle('Numerical Feature Boxplot (Outlier Detection)', fontsize=13)
plt.tight_layout()
plt.show()

# **5. Data Preprocessing**

In this stage, data preprocessing is a crucial step to ensure data quality before using it in a machine learning model.

Raw data often contains missing values, duplicates, or inconsistent value ranges, which can affect model performance. Therefore, this process aims to clean and prepare the data so that analysis runs optimally.

Here are the stages that can be performed, but are **not limited** to:
1. Removing or Handling Missing Values
2. Removing Duplicate Data
3. Normalizing or Standardizing Features
4. Outlier Detection and Handling
5. Encoding Categorical Data
6. Binning (Data Grouping)

Please adjust to the characteristics of the data you are using. Especially when using unstructured data.

In [27]:
print("Missing values before handling:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Fill missing values with median (robust to outliers)
df['ca'].fillna(df['ca'].median(), inplace=True)
df['thal'].fillna(df['thal'].median(), inplace=True)

print("\nMissing values after handling:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("No missing values!" if df.isnull().sum().sum() == 0 else "")

Missing values before handling:
ca      4
thal    2
dtype: int64

Missing values after handling:
Series([], dtype: int64)
No missing values!


In [28]:
print(f"Number of duplicates: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)
print(f"Shape after removing duplicates: {df.shape}")

Number of duplicates: 0
Shape after removing duplicates: (303, 14)


In [29]:
def remove_outliers_iqr(df, columns):
    df_clean = df.copy()
    for col in columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        before = len(df_clean)
        df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]
        after = len(df_clean)
        print(f"{col}: {before - after} rows deleted")
    return df_clean

df = remove_outliers_iqr(df, numerical_cols)
print(f"\nShape after handling outliers: {df.shape}")

age: 0 rows deleted
trestbps: 9 rows deleted
chol: 5 rows deleted
thalach: 1 rows deleted
oldpeak: 4 rows deleted

Shape after handling outliers: (284, 14)


In [30]:
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print(f"Shape after encoding: {df_encoded.shape}")
df_encoded.head()

Shape after encoding: (284, 21)


In [31]:
scaler = StandardScaler()
df_encoded[numerical_cols] = scaler.fit_transform(df_encoded[numerical_cols])

print("Scaling finished. Numerical feature statistics after scaling:")
df_encoded[numerical_cols].describe().round(3)

Scaling finished. Numerical feature statistics after scaling:


In [32]:
X = df_encoded.drop('target', axis=1)
y = df_encoded['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"y_train shape : {y_train.shape}")
print(f"y_test shape  : {y_test.shape}")

X_train shape : (227, 20)
X_test shape  : (57, 20)
y_train shape : (227,)
y_test shape  : (57,)


In [33]:
# Combine again for saving
train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

train_df.to_csv('heart_disease_preprocessing_train.csv', index=False)
test_df.to_csv('heart_disease_preprocessing_test.csv', index=False)

# Or save everything as one file
df_encoded.to_csv('heart_disease_preprocessing.csv', index=False)

print("Preprocessing dataset saved successfully!")
print(f"File: heart_disease_preprocessing.csv ({df_encoded.shape[0]} rows, {df_encoded.shape[1]} columns)")

Preprocessing dataset saved successfully!
File: heart_disease_preprocessing.csv (284 rows, 21 columns)
